# Lesson 80 · Vector similarity search — finding a kindred song among 100 million: cosine similarity and a pure-NumPy k-NN

**Goal**: first pin down what "similar" means, then use L2-normalized embeddings to run a cosine-similarity search and find the k most similar songs.

> Back-link to L10: get the relationship between the dot product, cosine similarity and L2 distance straight first, and the retrieval that follows falls into place.

🔗 **Aurora connection**: the `find_similar()` you build here compares a query embedding against the library matrix and returns the top k songs that are "most like you"; it is also the retrieval engine underneath `aurora.music.recommend`, and when FAISS arrives in Month 5 only this one function needs replacing.

← **Previous lesson** · [L79 · Music embeddings](L79_music_embed.ipynb)

> Last lesson covered **music embeddings**: contrastive learning pulls similar styles together and pushes different genres apart.
> This lesson explores **vector similarity search**.

## This lesson's story: how does Spotify actually rank its "You might also like"?

Every song goes through last lesson's MusicEncoder and comes out as a 128-dimensional vector. Now Spotify has 100 million songs, a user has just played a Beatles track, and the system has to find the 10 most similar songs to recommend within 0.1 seconds.

**The key question**: how do you measure how "similar" two vectors are?

**Three distances, three intuitions**:

| Metric | Formula | What it measures… | When to use it |
|---|---|---|---|
| Cosine similarity | cos(a,b) = a·b / (‖a‖·‖b‖) | the angle between directions | length doesn't matter, only style |
| L2 distance | ‖a-b‖ | distance in space | absolute position matters |
| Dot product | a·b | direction + length | after L2 normalization = cosine |

**The important conclusion**: after L2 normalization, **dot product = cosine similarity**. Trained embeddings are usually L2-normalized, so dot-product search = cosine search, and a matrix multiply computes it efficiently in one batch.

**The naive k-NN search**: compute the similarity between the query and all N vectors, then take the top k.
- Complexity: O(N×d), too slow when N = 100 million
- Approximate nearest neighbors (FAISS): compress with an index, roughly O(√N), at the cost of a little accuracy

**This lesson's task**: implement `find_similar()` in pure NumPy and verify that cosine-similarity search is correct.

> **🌉 Bridge**: last lesson (L79) you squeezed every song into one vector; this lesson asks just one thing — how alike are two vectors? And you already derived the measure of "alike" yourself back in **[L10 · The dot product](../2_linear_algebra/L10_dot_product.ipynb)**: a large dot product = the directions agree. This lesson's cosine similarity is nothing more than normalizing the vectors first and then taking L10's dot product.

In [ ]:
import numpy as np
import time

## 1. Cosine similarity and unit vectors

Cosine similarity is defined as:

```
cos_sim(a, b) = dot(a, b) / (||a|| * ||b||)
```

Its range is `[-1, 1]`: 1 means exactly the same direction (most similar), -1 means opposite directions, and 0 means orthogonal (unrelated).

**The key trick**: if you L2-normalize every vector first (turn it into a unit vector, ||v|| = 1), then `cos_sim(a, b) = dot(a, b)` — one matrix multiply does it all, with no dividing by lengths each time.

## 1b. L2 distance (Euclidean distance) compared with cosine similarity

L2 distance (Euclidean distance) is defined as:

```
L2(a, b) = ||a - b|| = sqrt(sum((a_i - b_i)^2))
```

Its range is `[0, +∞)`: 0 means identical, and the larger it is, the less similar.

**The essential difference between cosine similarity and L2 distance**:

| Measure | What it pays attention to | Good for |
|------|----------|----------|
| Cosine similarity | the vector's **direction** (ignores length) | text/audio embeddings, where direction carries meaning |
| L2 distance | the **absolute distance** between vectors (direction + length) | pixel distances, physical coordinates, differences in recommendation scores |

**The key link**: if the vectors are already L2-normalized (||v|| = 1), then:

```
L2(a, b)^2 = 2 - 2 * cos_sim(a, b)
```

In other words, after normalization, maximizing cosine similarity is the same as minimizing L2 distance.
This lesson takes the cosine-similarity route, because an embedding's length usually means nothing (only its direction carries meaning).


In [ ]:
# Demo: after normalization, cosine similarity and L2 distance are equivalent
a = np.array([3.0, 4.0])   # length 5
b = np.array([1.0, 0.0])   # length 1

# L2 distance (raw vectors)
l2_raw = np.linalg.norm(a - b)

# Verify the equivalence after normalization
a_n = a / np.linalg.norm(a)
b_n = b / np.linalg.norm(b)
cos_sim_val = np.dot(a_n, b_n)
l2_normalized = np.linalg.norm(a_n - b_n)
l2_from_cos = np.sqrt(2 - 2 * cos_sim_val)

print(f"L2(a,b) raw         = {l2_raw:.4f}")
print(f"L2(a_n, b_n)        = {l2_normalized:.4f}")
print(f"sqrt(2-2*cos_sim)   = {l2_from_cos:.4f}")
assert np.isclose(l2_normalized, l2_from_cos), "equivalence check failed"
print("\n✅ Verified: after normalization, L2^2 = 2 - 2*cos_sim")
print(f"   cos_sim={cos_sim_val:.4f}  L2_norm^2={l2_normalized**2:.4f}  2-2*cos={2-2*cos_sim_val:.4f}")


In [ ]:
# Demo: after L2 normalization, dot product == cosine similarity
a = np.array([3.0, 4.0])   # length 5
b = np.array([1.0, 0.0])   # length 1

# Raw cosine similarity
cos_raw = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Dot product after normalization
a_n = a / np.linalg.norm(a)
b_n = b / np.linalg.norm(b)
cos_normed = np.dot(a_n, b_n)

print(f'Raw cosine similarity:          {cos_raw:.4f}')
print(f'Dot product after normalizing:  {cos_normed:.4f}')
print(f'Equal: {np.isclose(cos_raw, cos_normed)}')

## 2. Brute-force top-k: argpartition vs argsort

There are two ways to find the k largest of N scores:

```
argsort:        O(N log N)  — a full sort, returns the complete ranking
argpartition:   O(N)        — only guarantees the top k land in the last k slots, unsorted inside
```

When N is large (a million songs) and k is small (return 5 songs), `argpartition` is clearly faster. Usage: `np.argpartition(scores, -k)[-k:]` gets the indices of the top k (in arbitrary order), then `scores[idx].argsort()[::-1]` sorts just those k.

In [ ]:
# Demo: argpartition vs argsort speed
rng = np.random.default_rng(42)
N, k = 100_000, 5
scores = rng.random(N)

t0 = time.perf_counter()
for _ in range(200):
    idx_sort = np.argsort(scores)[-k:][::-1]
t_sort = (time.perf_counter() - t0) / 200 * 1e6

t0 = time.perf_counter()
for _ in range(200):
    idx_part = np.argpartition(scores, -k)[-k:]
    idx_part = idx_part[np.argsort(scores[idx_part])[::-1]]
t_part = (time.perf_counter() - t0) / 200 * 1e6

print(f'argsort      average: {t_sort:.1f} µs')
print(f'argpartition average: {t_part:.1f} µs')
print(f'Same result: {np.array_equal(idx_sort, idx_part)}')

## 3. FAISS: approximate nearest neighbor (ANN) search at scale

[FAISS](https://github.com/facebookresearch/faiss) is a vector-search library from Meta. It supports index structures such as IVF and HNSW and still returns results in milliseconds at the million scale.

This lesson starts with a brute-force `np.dot` implementation, with the interface fixed as `(indices, scores)`. In Month 5 you only swap the function body for FAISS's `index.search()`, and the code above it doesn't change at all.

```
Brute-force search (this lesson):  fine for N < 100k, no extra dependencies
FAISS (Month 5):                   for N > 100k, needs pip install faiss-cpu
```

In [ ]:
# Demo: a preview of the FAISS interface (only installed for real in Month 5; here we just look at the calls)
print('Brute-force search interface:')
print('  indices, scores = find_similar(query_emb, library_embs, top_k=5)')
print()
print('FAISS interface (Month 5):')
print('  import faiss')
print('  index = faiss.IndexFlatIP(d)  # inner product = cosine (already normalized)')
print('  index.add(library_embs)')
print('  scores, indices = index.search(query_emb[None], top_k)')

## 4. ✏️ Implement `find_similar(query_emb, library_embs, top_k=5)`

**Signature**:
```python
def find_similar(
    query_emb: np.ndarray,          # (d,) query vector
    library_embs: np.ndarray,       # (N, d) song library
    top_k: int = 5
) -> tuple[np.ndarray, np.ndarray]:
    # returns: (indices, scores)
    # indices: (top_k,) int, indices sorted from high to low
    # scores: (top_k,) float, the matching cosine similarities
```

**A 4-step route**:

| Step | Operation | NumPy tool |
|---|---|---|
| 1 | L2-normalize the query | `q = query_emb / np.linalg.norm(query_emb)` |
| 2 | L2-normalize the library (row by row) | `lib = library_embs / np.linalg.norm(library_embs, axis=1, keepdims=True)` |
| 3 | Batched dot product (= cosine similarity) | `scores = lib @ q` → shape (N,) |
| 4 | Top-k: `argpartition` + `argsort` | `np.argpartition(-scores, top_k)[:top_k]` |

**Acceptance criteria**:
- Orthogonal embeddings (`np.eye(4)`): query[0]'s nearest neighbor is itself (score=1.0)
- Similarities come out in descending order
- Returns indices and scores, both of shape (top_k,)

In [ ]:
def find_similar(
    query_emb: np.ndarray,
    library_embs: np.ndarray,
    top_k: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    """Top-k search by cosine similarity.

    Parameters
    ----
    query_emb   : shape (d,)   query vector
    library_embs: shape (N, d) song-library embeddings
    top_k       : return the k most similar songs

    Returns
    ----
    indices : shape (top_k,)  row indices into library_embs, by descending similarity
    scores  : shape (top_k,)  the matching cosine similarities
    """
    # ✏️ TODO 1: L2-normalize query_emb -> q
    # q = ...

    # ✏️ TODO 2: L2-normalize each row of library_embs -> L
    # L = ...

    # ✏️ TODO 3: scores = L @ q, shape (N,)
    # scores = ...

    # ✏️ TODO 4: argpartition to get top_k candidates, then sort them descending
    # top_k = min(top_k, len(scores))
    # part_idx = ...
    # order = ...
    # indices = ...

    raise NotImplementedError("Please implement find_similar")

In [ ]:
# Basic correctness check
embs = np.eye(4)  # 4 songs, mutually orthogonal
try:
    result = find_similar(embs[0], embs, top_k=2)
except (NotImplementedError, TypeError):
    result = None

if result is None or result is ...:
    print("⬜ find_similar not implemented yet")
else:
    idx, s = result
    assert 0 in idx, f'index 0 (the exact match) must be in the top 2, got {idx}'
    assert np.isclose(s[0], 1.0), f'the top score should be 1.0, got {s[0]:.4f}'
    assert s[0] >= s[1], 'scores should be in descending order'
    print('✅ Basic check passed')

    # Check the returned shapes
    rng = np.random.default_rng(0)
    big_lib = rng.standard_normal((100, 32))
    q = rng.standard_normal(32)
    idx2, s2 = find_similar(q, big_lib, top_k=10)
    assert idx2.shape == (10,), f'indices shape should be (10,), got {idx2.shape}'
    assert s2.shape == (10,), f'scores shape should be (10,), got {s2.shape}'
    assert all(s2[i] >= s2[i+1] for i in range(9)), 'scores are not in descending order'
    print('✅ Shape and ordering checks passed')

## 5. Parameter experiment: a 50-song library, 10 queries

**Parameters**: `N=50` (number of songs), `d=128` (embedding dimension), `top_k=5`

**What you should see**:
- Each search takes < 1 ms (a brute-force dot product is extremely fast at N=50)
- `scores` lie in `[-1, 1]`, and the top score is close to 1 (if the query comes from the library)
- Turning `top_k` up to 50 still gives correct results (it returns the ranking of every song)

In [ ]:
rng = np.random.default_rng(7)
N, d, top_k = 50, 128, 5
library = rng.standard_normal((N, d))

times = []
for i in range(10):
    query = library[i]  # use song i from the library as the query; expect it to score highest against itself
    t0 = time.perf_counter()
    try:
        result = find_similar(query, library, top_k=top_k)
    except (NotImplementedError, TypeError):
        print("⬜ find_similar not implemented yet"); break
    if result is None:
        print("⬜ find_similar not implemented yet"); break
    idx, scores = result
    elapsed_us = (time.perf_counter() - t0) * 1e6
    times.append(elapsed_us)
    top1_is_self = idx[0] == i
    print(f'query={i:2d}  top1={idx[0]:2d}(self={top1_is_self})  '
          f'score={scores[0]:.4f}  time={elapsed_us:.1f}µs')

if not times:
    print('⬜ find_similar not implemented yet, skipping the speed assertion')
else:
    avg_ms = np.mean(times) / 1000
    print(f'\nAverage search time: {avg_ms:.3f} ms  (should be < 1 ms for N={N})')
    assert avg_ms < 1.0, f'search too slow: {avg_ms:.3f} ms'

## Wrap-up

`find_similar()` takes a query embedding and a song-library matrix and returns an `(indices, scores)` pair sorted by descending cosine similarity. It will serve as the retrieval engine underneath `aurora.music.recommend`: the layer above just passes in embeddings produced by the encoder and never needs to know how the search works. Once FAISS comes in at Month 5, only this function's insides change and the interface stays the same — next lesson, L81, builds a complete playlist-recommendation pipeline on top of it.

## ✏️ Whiteboard challenge: similarity search by hand (target: 10 minutes)

Cover the screen and work it out on paper:

**Question 1**: a=[3,4], b=[1,0]. Cosine similarity = ? L2 distance = ?
(cos=a·b/(‖a‖·‖b‖)=(3)/(5×1)=0.6; L2=‖[2,4]‖=√20≈4.47)

**Question 2**: after L2 normalization a'=[0.6,0.8], b'=[1,0]. Verify: cosine similarity = dot product?
(a'·b'=0.6×1+0.8×0=0.6; cos(a',b')=a'·b'/(‖a'‖·‖b'‖)=0.6/(1×1)=0.6 ✓)

**Question 3**: N=100 million, d=128. How many multiplications does one brute-force k-NN query need?
(N×d=10^8×128=1.28×10^10; at 1 ns each that's 12.8 seconds — too slow)

**Question 4**: `argpartition` vs `argsort` for finding the top 5: what is the time complexity of each?
(argpartition: O(N), only guarantees the right top k, unsorted; argsort: O(N log N), a full sort; for large N argpartition is faster)

**Question 5**: if the query vector is all zeros (the zero vector), what happens under L2 normalization? How should you handle it?
(0/‖0‖=0/0=nan; check norm>1e-8 before normalizing, or return a random result / an empty list)

Once you've worked them out, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: cosine similarity and L2 distance
a_q1 = np.array([3.0, 4.0])
b_q1 = np.array([1.0, 0.0])
cos_q1 = np.dot(a_q1, b_q1) / (np.linalg.norm(a_q1) * np.linalg.norm(b_q1))
l2_q1 = np.linalg.norm(a_q1 - b_q1)
assert abs(cos_q1 - 0.6) < 1e-6, f"cos should be 0.6, got {cos_q1:.4f}"
assert abs(l2_q1 - np.sqrt(20)) < 1e-6, f"L2 should be √20, got {l2_q1:.4f}"
print(f"Q1 ✅  cos([3,4],[1,0])={cos_q1:.4f}, L2={l2_q1:.4f} (=√20≈{np.sqrt(20):.4f})")

# Q2: after L2 normalization, dot product = cosine
a_n = a_q1 / np.linalg.norm(a_q1)
b_n = b_q1 / np.linalg.norm(b_q1)
dot_normalized = np.dot(a_n, b_n)
cos_normalized = np.dot(a_n, b_n) / (np.linalg.norm(a_n) * np.linalg.norm(b_n))
assert abs(dot_normalized - cos_normalized) < 1e-9, "after normalization the dot product should equal the cosine"
assert abs(dot_normalized - 0.6) < 1e-6, f"should be 0.6, got {dot_normalized:.4f}"
print(f"Q2 ✅  after normalization: dot product={dot_normalized:.4f}, cosine={cos_normalized:.4f} (exactly equal)")

# Q3: number of multiplications in brute-force k-NN
N_q3, d_q3 = 10**8, 128
ops_q3 = N_q3 * d_q3
print(f"Q3 ✅  N=100 million, d=128: multiplications={ops_q3:.2e} (at 1 ns each: {ops_q3*1e-9:.1f} s, too slow → you need FAISS)")

# Q4: argpartition vs argsort
rng_q4 = np.random.default_rng(42)
scores_q4 = rng_q4.standard_normal(100000)
k_q4 = 5
# argpartition: O(N)
idx_part = np.argpartition(-scores_q4, k_q4)[:k_q4]
top_part = scores_q4[idx_part]
# argsort: O(N log N)
idx_sort = np.argsort(-scores_q4)[:k_q4]
top_sort = scores_q4[idx_sort]
assert set(idx_part) == set(idx_sort[:k_q4]), "both methods should return the same set"
print(f"Q4 ✅  argpartition set={sorted(scores_q4[idx_part])[::-1][:2]}..., argsort result={sorted(scores_q4[idx_sort])[::-1][:2]}... (same set)")

# Q5: handling the zero vector
# L2-normalizing the zero vector = 0/0 = nan; the reference implementation adds 1e-12 to the
# denominator to avoid nan, so a zero-vector query still returns safely (all similarities 0, no crash).
try:
    zero_query = np.zeros(128)
    norm_val = np.linalg.norm(zero_query)
    if norm_val < 1e-8:
        print(f"Q5 ✅  zero vector norm={norm_val:.2e}<1e-8; naive normalization gives nan (add an eps or skip it)")
    else:
        print(f"Q5 ✅  vector norm={norm_val:.4f}, normalizes normally")
    # Query the library with a zero vector of matching dimension (query and library both 4-D) to check it returns safely
    zq = np.zeros(4)
    idx_z, sc_z = find_similar(zq, np.eye(4), top_k=2)
    assert not np.isnan(sc_z).any(), "a zero vector should not produce nan scores"
    print(f"Q5 ✅  find_similar(zero vector) returned safely: indices={idx_z}, scores[0]={sc_z[0]:.4f} (no nan)")
except (NotImplementedError, TypeError):
    print(f"Q5 ✅  zero vector → nan; add an eps before normalizing or check norm>1e-8 (verify once implemented)")

print("\n🎉 Similarity-search whiteboard challenge passed!")


In [ ]:
# ✏️ Self-review for this lesson
l80_review = {
    "cosine_vs_l2_understood":  None,  # understand how cos / L2 / dot product differ, and that after L2 normalization dot product = cosine? True/False
    "topk_complexity":          None,  # understand argpartition O(N) vs argsort O(N log N)? True/False
    "find_similar_impl":        None,  # find_similar() is correct: orthogonal embeddings → self-similarity = 1? True/False
    "zero_vector_edge":         None,  # understand the zero-vector normalization problem and how to handle it? True/False
    "whiteboard_passed":        None,  # worked all 5 whiteboard questions by hand? True/False
}

unfilled = [k for k, v in l80_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l80_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L80 all clear! On to L81: a music recommender')

---

→ **Next lesson** · [L81 · A music recommender](L81_recommendation.ipynb)

> Next lesson covers **a music recommender**: user taste → embeddings → k-NN neighbors → a recommendation list.